# 8. SAE Training → Manifold Figure
## Trains one SAE on the gecko NCA recording, then saves `figs/sae_manifold.png`.
### Harald Michael Ludwig
### April 2026


Code to support ALIFE 2026 paper:
 > Stovold, Kvalsund, Ludwig, Sharma, Mordvintsev "Visualising the Attractor Landscape of Neural Cellular Automata"

Code and materials are restricted for any and all use prior to publication; post-publication the code and associated materials are available under the Apache 2.0 license


In [ ]:
DATA_PATH = "bluegreenrepeat.npy"
OUT_DIR = "figs"

H, W, C, T = 60, 60, 17, 15000

SAE_CFG = dict(
    l1_coef=3e-2,
    batch_size=4096,
    expansion=64,
    n_epochs=30,
    lr=1e-3,
    seed=0,
)

In [ ]:
import os, sys
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
from typing import NamedTuple
from jaxtyping import Array, Float
from sklearn.decomposition import PCA

In [ ]:
raw = np.load(DATA_PATH, allow_pickle=True)  # (1, 60, 60, 17, 15000)
print(f"Raw data: {raw.shape}")

# Flatten all cells from all frames → training corpus
cells = raw[0].transpose(3, 0, 1, 2).reshape(T * H * W, C).astype(np.float32)
print(f"Training set: {cells.shape}  ({cells.nbytes / 1e9:.2f} GB)")

In [ ]:
class SAE(eqx.Module):
    W_enc: Float[Array, "d_in d_hidden"]
    W_dec: Float[Array, "d_hidden d_in"]
    b_enc: Float[Array, "d_hidden"]
    b_dec: Float[Array, "d_in"]

    def __init__(self, d_in, d_hidden, *, key):
        k1, k2 = jax.random.split(key)
        self.W_enc = jax.random.normal(k1, (d_in, d_hidden)) * jnp.sqrt(2.0 / d_in)
        W_dec = jax.random.normal(k2, (d_hidden, d_in)) * jnp.sqrt(1.0 / d_hidden)
        self.W_dec = W_dec / jnp.linalg.norm(W_dec, axis=1, keepdims=True)
        self.b_enc = jnp.zeros(d_hidden)
        self.b_dec = jnp.zeros(d_in)

    def encode(self, x):
        return jax.nn.relu((x - self.b_dec) @ self.W_enc + self.b_enc)

    def __call__(self, x):
        z = self.encode(x)
        return z @ self.W_dec + self.b_dec, z


class Metrics(NamedTuple):
    loss: float
    mse: float
    l1: float
    l0: float
    dead_frac: float


def sae_loss(sae, x, l1_coef):
    x_hat, z = sae(x)
    mse = jnp.mean((x - x_hat) ** 2)
    l1 = jnp.mean(jnp.abs(z))
    l0 = jnp.mean(jnp.sum(z > 0, axis=-1).astype(float))
    dead = jnp.mean(jnp.all(z == 0, axis=0).astype(float))
    return mse + l1_coef * l1, Metrics(mse + l1_coef * l1, mse, l1, l0, dead)


@eqx.filter_jit
def train_step(sae, opt_state, x, opt, l1_coef):
    (_, metrics), grads = eqx.filter_value_and_grad(sae_loss, has_aux=True)(
        sae, x, l1_coef
    )
    updates, opt_state = opt.update(grads, opt_state, sae)
    sae = eqx.apply_updates(sae, updates)
    # Re-normalise decoder columns
    W_dec = sae.W_dec / jnp.linalg.norm(sae.W_dec, axis=1, keepdims=True)
    return eqx.tree_at(lambda s: s.W_dec, sae, W_dec), opt_state, metrics

## Train SAE

In [ ]:
cfg = SAE_CFG
d_hidden = C * cfg["expansion"]
key = jax.random.PRNGKey(cfg["seed"])

sae = SAE(C, d_hidden, key=key)
opt = optax.adam(cfg["lr"])
opt_state = opt.init(eqx.filter(sae, eqx.is_array))

n, bs = len(cells), cfg["batch_size"]
n_batches = n // bs
print(
    f"SAE: {C} → {d_hidden}   l1={cfg['l1_coef']:.0e}  bs={bs}  epochs={cfg['n_epochs']}"
)
print(f"{n:,} cells  ×  {n_batches} batches/epoch")

for epoch in range(cfg["n_epochs"]):
    key, sk = jax.random.split(key)
    perm = np.array(jax.random.permutation(sk, n))
    data_s = cells[perm]
    epoch_m = []

    for i in range(n_batches):
        batch = jax.device_put(data_s[i * bs : (i + 1) * bs])
        sae, opt_state, m = train_step(sae, opt_state, batch, opt, cfg["l1_coef"])
        epoch_m.append(m)

    mse = float(np.mean([m.mse for m in epoch_m]))
    l0 = float(np.mean([m.l0 for m in epoch_m]))
    dead = float(np.mean([m.dead_frac for m in epoch_m]))
    print(f"  epoch {epoch + 1:3d}  mse={mse:.2e}  L0={l0:.1f}  dead={dead:.1%}")
    sys.stdout.flush()

print("Training done.")

In [ ]:
# Encode all frames (mean-pool over cells)
print("Encoding all frames:")
Z_frames = np.zeros((T, d_hidden), dtype=np.float32)

for t in range(T):
    frame = raw[0, :, :, :, t].astype(np.float32)  # (H, W, C)
    cells_t = frame.reshape(-1, C)
    _, z = sae(jax.device_put(cells_t))
    Z_frames[t] = np.array(z).mean(axis=0)
    if t % 3000 == 0:
        print(f"  {t}/{T}")

print(f"Done.\nZ_frames: {Z_frames.shape}")

## Create figure 10 (PCA)

In [ ]:
frames_t = raw[0].transpose(3, 0, 1, 2)  # (T, H, W, C)
colours = frames_t[..., :3].mean(axis=(1, 2))  # (T, 3)
colours = (colours - colours.min()) / colours.max()
colours = np.clip(colours, 0, 1)

print("PCA:")
pca = PCA(n_components=50, random_state=42)
Zf_pca = pca.fit_transform(Z_frames)
pc_var = pca.explained_variance_ratio_
print(f"PC1: {pc_var[0]:.1%}   PC2: {pc_var[1]:.1%}")

In [ ]:
mpl.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 9,
        "axes.titlesize": 9,
        "axes.labelsize": 9,
        "legend.fontsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "axes.linewidth": 0.6,
        "lines.linewidth": 0.8,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

Z2 = Zf_pca[:, :2]

fig, ax = plt.subplots(1, 1, figsize=(3.4, 3.4))

ax.plot(Z2[:, 0], Z2[:, 1], c="k", lw=0.15, alpha=0.12, zorder=0)
ax.scatter(
    Z2[:, 0],
    Z2[:, 1],
    c=colours,
    s=2.0,
    alpha=0.65,
    rasterized=True,
    zorder=2,
    linewidths=0,
)

marker_kw = dict(facecolor="white", edgecolor="black", linewidths=0.8)
ax.scatter(*Z2[0], marker="o", s=55, zorder=7, label=r"$t=0$", **marker_kw)
ax.scatter(*Z2[T // 2], marker="s", s=45, zorder=6, label=rf"$t={T // 2}$", **marker_kw)
ax.scatter(*Z2[-1], marker="X", s=60, zorder=5, label=rf"$t={T - 1}$", **marker_kw)

ax.set_xlabel(f"PC1 ({pc_var[0]:.0%})")
ax.set_ylabel(f"PC2 ({pc_var[1]:.0%})")
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect("equal", adjustable="datalim")
ax.legend(loc="best", frameon=True, framealpha=0.85, handletextpad=0.3, borderpad=0.3)

plt.tight_layout()

os.makedirs(OUT_DIR, exist_ok=True)
png_path = os.path.join(OUT_DIR, "sae_manifold.png")
fig.savefig(png_path, dpi=400)
print(f"Saved: {png_path}")
plt.show()